# Feature engineering

**Philosophy:** Build **team-level** stats per (Season, TeamID) from regular-season data (win rate, PPG, PAPG, efficiency, seed, etc.). Then in the matchup layer we use **differences** (Team A stat minus Team B stat)—never raw team stats—to keep features comparable and avoid leakage.

Reloading feature module

In [50]:
import sys
sys.path.insert(0, '..')
import importlib
import src.features
import src.data_loading

importlib.reload(src.features)
importlib.reload(src.data_loading)



<module 'src.data_loading' from '/Users/calebrh/model-madness/march-machine-learning-mania-2026/notebooks/../src/data_loading.py'>

Loading Data

In [34]:
from src.data_loading import load_regular_season_results

games = load_regular_season_results(compact=False)

In [19]:
from src import features

# Stub: build_team_season_stats expects DataFrames from data_loading
# Expected output columns: Season, TeamID, win_pct, ppg, papg, ... (see features.get_feature_columns)
try:
    features.build_team_season_stats(None)
except NotImplementedError:
    print("build_team_season_stats is a stub; implement with regular_season_results, seeds, optional ratings.")
try:
    features.get_feature_columns()
except NotImplementedError:
    print("get_feature_columns will return the list of feature names for matchup diffs.")

build_team_season_stats is a stub; implement with regular_season_results, seeds, optional ratings.
get_feature_columns will return the list of feature names for matchup diffs.


Building the

Adding winner and loser rows

In [20]:
from src.game_results import make_long_regular_season_results

team_games = make_long_regular_season_results(games)
print("Long-format regular-season rows:", team_games.shape)

team_games.head()


Long-format regular-season rows: (249058, 33)


,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_fga3,opp_ftm,opp_fta,opp_or,opp_dr,opp_ast,opp_to,opp_stl,opp_blk,opp_pf
0,2003,10,1104,1328,1,68,62,27,58,3,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,1393,1,70,63,26,62,8,...,24,9,20,20,25,7,12,8,6,16
2,2003,10,1328,1104,0,62,68,22,53,2,...,14,11,18,14,24,13,23,7,1,22
3,2003,10,1393,1272,0,63,70,24,67,6,...,20,10,19,15,28,16,13,4,4,18
4,2003,11,1186,1458,0,55,81,20,46,3,...,12,23,27,12,24,12,9,9,3,18


Sanity check for winner and loser row aggregations

In [21]:
team_games.shape
team_games.head()

,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_fga3,opp_ftm,opp_fta,opp_or,opp_dr,opp_ast,opp_to,opp_stl,opp_blk,opp_pf
0,2003,10,1104,1328,1,68,62,27,58,3,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,1393,1,70,63,26,62,8,...,24,9,20,20,25,7,12,8,6,16
2,2003,10,1328,1104,0,62,68,22,53,2,...,14,11,18,14,24,13,23,7,1,22
3,2003,10,1393,1272,0,63,70,24,67,6,...,20,10,19,15,28,16,13,4,4,18
4,2003,11,1186,1458,0,55,81,20,46,3,...,12,23,27,12,24,12,9,9,3,18


Building advanced features for the team games (can go multiple different directions from here)

In [22]:
from src.features import build_advanced_features

advanced_features = build_advanced_features(team_games)
advanced_features.head()

,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_pf,margin,possessions,efg_pct,tov_pct,orb_pct,ftr,off_eff,def_eff,net_eff
0,2003,10,1104,1328,1,68,62,27,58,3,...,20,6,73.5000,0.491379,0.312925,0.388889,0.310345,92.517007,84.353741,8.163265
1,2003,10,1272,1393,1,70,63,26,62,8,...,16,7,68.7625,0.483871,0.189057,0.375000,0.306452,101.799673,91.619706,10.179967
2,2003,10,1328,1104,0,62,68,22,53,2,...,22,-6,73.5000,0.433962,0.244898,0.294118,0.415094,84.353741,92.517007,-8.163265
3,2003,10,1393,1272,0,63,70,24,67,6,...,18,-7,68.7625,0.402985,0.174514,0.416667,0.298507,91.619706,101.799673,-10.179967
4,2003,11,1186,1458,0,55,81,20,46,3,...,18,-26,66.9500,0.467391,0.283794,0.200000,0.369565,82.150859,120.985810,-38.834951


Getting season long regular season stats

In [23]:
from src.features import build_regular_season_team_stats

season_features = build_regular_season_team_stats(advanced_features)
season_features.head()

,Season,TeamID,games_played,win_pct,avg_margin,avg_points_for,avg_points_against,off_eff,def_eff,net_eff,efg_pct,tov_pct,orb_pct,ftr
0,2003,1102,28,0.428571,0.250000,57.250000,57.000000,103.835749,103.601436,0.234313,0.584407,0.205580,0.168235,0.446693
1,2003,1103,27,0.481481,0.629630,78.777778,78.148148,110.667469,110.408760,0.258709,0.536564,0.179502,0.305803,0.465135
2,2003,1104,28,0.607143,4.285714,69.285714,65.000000,103.557697,97.794103,5.763594,0.475785,0.199612,0.371256,0.372350
3,2003,1105,26,0.269231,-4.884615,71.769231,76.653846,93.616492,100.207433,-6.590941,0.457983,0.242292,0.335166,0.359501
4,2003,1106,28,0.464286,-0.142857,63.607143,63.750000,93.773763,94.224204,-0.450441,0.481697,0.251113,0.349480,0.307563


Sanity check on season features

In [24]:
season_features.groupby(["Season", "TeamID"]).size().value_counts()
season_features.isna().sum()
season_features.describe()
season_features.sort_values("net_eff", ascending = False).head(20)

,Season,TeamID,games_played,win_pct,avg_margin,avg_points_for,avg_points_against,off_eff,def_eff,net_eff,efg_pct,tov_pct,orb_pct,ftr
5585,2019,1211,33,0.909091,23.787879,88.848485,65.060606,123.829359,90.759667,33.069693,0.594681,0.144751,0.301536,0.359958
4883,2017,1211,33,0.969697,23.424242,84.575758,61.151515,119.293332,86.274725,33.018607,0.580935,0.158583,0.291260,0.394115
4215,2015,1246,34,1.000000,20.941176,74.911765,53.970588,115.910391,83.659554,32.250837,0.520134,0.163519,0.402745,0.451499
7691,2025,1181,34,0.911765,20.794118,82.705882,61.911765,122.498919,91.694300,30.804619,0.574671,0.136420,0.335700,0.327781
3875,2014,1257,34,0.852941,21.147059,82.117647,60.970588,118.661179,88.063803,30.597376,0.540508,0.148240,0.370055,0.414850
6286,2021,1211,26,1.000000,23.000000,92.115385,69.115385,120.937119,90.584286,30.352834,0.610733,0.156097,0.271928,0.371953
6638,2022,1211,29,0.896552,22.482759,87.827586,65.344828,117.564555,87.544926,30.019629,0.594659,0.155430,0.269913,0.300106
3467,2013,1196,33,0.787879,17.939394,71.636364,53.696970,114.435507,85.672215,28.763292,0.558752,0.176417,0.338462,0.299035
1783,2008,1242,33,0.909091,19.515152,81.181818,61.666667,117.482390,89.292836,28.189554,0.561901,0.186586,0.379112,0.383291
7006,2023,1222,34,0.911765,18.529412,75.029412,56.500000,114.118478,85.987843,28.130635,0.526515,0.142771,0.368546,0.290498


Building recent features (stats for teams in x recent games (default 10))

In [25]:
from src.features import get_recent_features

recent_features = get_recent_features(team_games)
recent_features.head()

,Season,TeamID,lastx_win_pct,lastx_avg_margin,lastx_off_eff,lastx_def_eff,lastx_net_eff
0,2003,1102,0.2,-8.0,100.049778,114.969991,-14.920213
1,2003,1103,0.5,0.6,110.346843,109.392363,0.954480
2,2003,1104,0.4,1.1,107.131388,105.967896,1.163492
3,2003,1105,0.3,-0.2,99.156553,99.337066,-0.180513
4,2003,1106,0.4,0.1,92.946370,93.041995,-0.095626


Combine recent and season features

In [26]:
from src.features import combine_features

combined_features = combine_features(season_features, recent_features)
combined_features.head()

,Season,TeamID,games_played,win_pct,avg_margin,avg_points_for,avg_points_against,off_eff,def_eff,net_eff,efg_pct,tov_pct,orb_pct,ftr,lastx_win_pct,lastx_avg_margin,lastx_off_eff,lastx_def_eff,lastx_net_eff
0,2003,1102,28,0.428571,0.250000,57.250000,57.000000,103.835749,103.601436,0.234313,0.584407,0.205580,0.168235,0.446693,0.2,-8.0,100.049778,114.969991,-14.920213
1,2003,1103,27,0.481481,0.629630,78.777778,78.148148,110.667469,110.408760,0.258709,0.536564,0.179502,0.305803,0.465135,0.5,0.6,110.346843,109.392363,0.954480
2,2003,1104,28,0.607143,4.285714,69.285714,65.000000,103.557697,97.794103,5.763594,0.475785,0.199612,0.371256,0.372350,0.4,1.1,107.131388,105.967896,1.163492
3,2003,1105,26,0.269231,-4.884615,71.769231,76.653846,93.616492,100.207433,-6.590941,0.457983,0.242292,0.335166,0.359501,0.3,-0.2,99.156553,99.337066,-0.180513
4,2003,1106,28,0.464286,-0.142857,63.607143,63.750000,93.773763,94.224204,-0.450441,0.481697,0.251113,0.349480,0.307563,0.4,0.1,92.946370,93.041995,-0.095626


Checking loading all team features method

In [69]:
from src.features import get_all_team_features
from src.data_loading import load_team_names

all_features = get_all_team_features()

#all_features = all_features[all_features["seed_num"].notna()]
print(all_features.shape)

#team_names = load_team_names()

#all_features = all_features.merge(team_names, on="TeamID", how="left")

from pathlib import Path

# Adjust this base path if your notebook runs from the repo root
PROJECT_ROOT = Path.cwd().parent  # or Path("march-machine-learning-mania-2026")

output_dir = PROJECT_ROOT / "data" / "interim"

all_features = all_features.sort_values(["Season", "seed_num"], ascending=False).reset_index(drop=True)
all_features.to_csv(output_dir / "men_team_features.csv", index=False)
# Display 30 rows from most recent season to least recent
all_features.head(30)

(8346, 20)


,Season,TeamID,games_played,win_pct,avg_margin,avg_points_for,avg_points_against,off_eff,def_eff,net_eff,efg_pct,tov_pct,orb_pct,ftr,lastx_win_pct,lastx_avg_margin,lastx_off_eff,lastx_def_eff,lastx_net_eff,seed_num
0,2026,1224,29,0.655172,7.206897,75.000000,67.793103,106.129994,95.870266,10.259728,0.509266,0.178934,0.347034,0.452563,0.9,19.0,116.848491,90.613500,26.234991,16.0
1,2026,1250,32,0.500000,-2.281250,72.375000,74.656250,104.022510,107.219966,-3.197456,0.534754,0.166458,0.208555,0.325955,0.8,2.7,111.501956,107.524453,3.977503,16.0
2,2026,1254,34,0.705882,2.970588,74.088235,71.117647,106.733961,102.090089,4.643872,0.533639,0.178805,0.330523,0.355069,0.8,7.1,110.365814,99.692784,10.673029,16.0
3,2026,1341,31,0.451613,-2.806452,75.741935,78.548387,102.583068,105.508895,-2.925828,0.480992,0.147559,0.238858,0.434909,0.9,9.2,105.540981,92.177578,13.363403,16.0
4,2026,1373,34,0.676471,4.852941,70.529412,65.676471,108.476907,100.721255,7.755652,0.509233,0.148598,0.301758,0.356639,0.6,2.0,103.909024,100.308464,3.600560,16.0
5,2026,1420,30,0.733333,7.466667,75.433333,67.966667,111.476501,100.338318,11.138183,0.538469,0.128970,0.226229,0.342941,1.0,18.2,119.559514,92.661406,26.898108,16.0
6,2026,1202,31,0.612903,2.903226,75.064516,72.161290,110.132065,105.637075,4.494990,0.550370,0.166544,0.290556,0.323256,0.6,3.8,111.881904,105.940421,5.941483,15.0
7,2026,1225,32,0.562500,2.656250,76.656250,74.000000,108.021775,104.431465,3.590310,0.521569,0.141228,0.255391,0.353090,0.8,9.2,110.612683,97.576011,13.036672,15.0
8,2026,1398,29,0.689655,4.137931,78.068966,73.931034,108.154602,101.822066,6.332537,0.501120,0.153253,0.321165,0.342906,0.9,13.8,114.318010,93.926359,20.391652,15.0
9,2026,1474,33,0.606061,1.666667,84.818182,83.151515,116.992615,114.599186,2.393429,0.567142,0.140315,0.280530,0.368477,0.8,6.3,117.946695,109.392524,8.554171,15.0


Adding matchup data